# 03 AgentSHAP run for the analysis in notebook 02

**Required: a local Ollama, or you can carefully wire up a paid API with the agent.**

```bash
ollama serve
ollama pull qwen2.5:3b
```

**Files produced by running the notebook:**
| File | What it contains  |
|---|---|
| `finance.csv`, `finance_shapley.csv` | one `analyze()` on a prompt with tools: coalitions, answers, calls, Shapley values |
| `noise_floor.json` | Similarity ceiling $\nu$ on three prompts with tools: how similar two answers on the **same** toolset are |
| `notool_coalitions.csv`, `notool_shapley.csv` | `analyze()` on five prompts where no tool is needed. Five is a synthetic small number; for stronger results you obviously need more. |
| `noise_floor_notool.json` | $\nu$ on each of these five prompts separately |

Generation is stochastic (`temperature=0.5` is hardcoded in the library), so
a repeated run will give different numbers. The conclusions of `02_agent_attribution.ipynb` do not depend on this.

The run takes a few minutes.

In [1]:
import sys; sys.path.insert(0, "..")
from pathlib import Path

import pandas as pd
import requests

from agent import MODEL_NAME, OLLAMA_URL, build_agent_shap, measure_noise_floor

OUT = Path("../data/agentshap")
OUT.mkdir(exist_ok=True)

models = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
assert MODEL_NAME in models, f"{MODEL_NAME} is not in Ollama: ollama pull {MODEL_NAME}. Available: {models}"

bundle = build_agent_shap()   # on first run, clones TokenSHAP next to the repository
agent_shap, tool_names = bundle.agent_shap, bundle.tool_names
print("model:", MODEL_NAME, "  tools:", tool_names)

model: qwen2.5:3b   tools: ['get_weather', 'get_stock_price', 'calculate', 'get_news']


## Example of `analyze()` on a prompt that needs tools

In [2]:
PROMPT = "What's the weather in New York and how is AAPL stock doing today?"
def run(prompt):
    # analyze() plus the baseline, which the library keeps in attributes, not in results_df
    res, sv = agent_shap.analyze(prompt=prompt, sampling_ratio=0.0)
    base = {"combination_key": "baseline", "tools_available": "+".join(tool_names),
            "response": agent_shap._baseline_response, "similarity": None,
            **{f"used_{k}": v for k, v in agent_shap._baseline_tool_usage.items()}}
    res = pd.concat([pd.DataFrame([base]), res.drop(columns="indexes")], ignore_index=True)
    return res.assign(prompt=prompt), {"prompt": prompt, **sv}

finance, finance_shap = run(PROMPT)
finance.to_csv(OUT / "finance.csv", index=False)
pd.DataFrame([finance_shap]).to_csv(OUT / "finance_shapley.csv", index=False)
print(finance.drop(columns=["response", "prompt"]).to_string(index=False))
print("\nshapley:", {k: round(v, 4) for k, v in finance_shap.items() if k != "prompt"})


Analyzing 4 tools:
  - get_weather
  - get_stock_price
  - calculate
  - get_news

1. Getting baseline response (all tools)...



2. Testing tool combinations...



Testing tool combinations:   0%|          | 0/4 [00:00<?, ?it/s]


Testing tool combinations:  25%|██▌       | 1/4 [00:13<00:40, 13.58s/it]


Testing tool combinations:  50%|█████     | 2/4 [00:22<00:22, 11.05s/it]


Testing tool combinations:  75%|███████▌  | 3/4 [00:30<00:09,  9.33s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:36<00:00,  8.35s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:36<00:00,  9.25s/it]


3. Calculating similarities...

4. Computing Shapley values...

 Analysis complete!
                               combination_key                                tools_available similarity  used_get_weather  used_get_stock_price  used_get_news
                                      baseline get_weather+get_stock_price+calculate+get_news       None               1.0                   1.0            NaN
   calculate+get_news+get_stock_price_idx2,3,4             calculate+get_news+get_stock_price   0.225646               NaN                   2.0            1.0
       calculate+get_news+get_weather_idx1,3,4                 calculate+get_news+get_weather   0.263532               1.0                   NaN            1.0
 get_news+get_stock_price+get_weather_idx1,2,4           get_news+get_stock_price+get_weather   0.624593               1.0                   1.0            NaN
calculate+get_stock_price+get_weather_idx1,2,3          calculate+get_stock_price+get_weather   0.683045           

## Noise floor

By construction, AgentSHAP compares the answer with a reduced tool set against a single answer with the full set: $v(S) = \cos(\text{answer}(S), \text{answer}(N))$, contribution $\propto 1 - v(N \setminus \{i\})$.

Here the formula assumes that $v(N) = 1$.

But usually we don't run models deterministically (and in general there is no determinism). Besides, the hyperparameter `temperature=0.5` is hardcoded in the library, and two runs with the **same** set diverge: $\nu = \mathbb{E}[\cos(\text{answer}(N), \text{answer}'(N))] < 1$.

The library smears the divergence $1 - \nu$ over all tools. That is, in an ideal world we want
- $\nu = \mathbb{E}[\cos(\text{answer}(N), \text{answer}'(N))] = 1$, and then $v(N) = 1$ holds.

in the ideal case we get that $v(N) = 1$ does not have to hold. When using the framework, ν is needed to replace the unreachable one, $\text{contribution}_i = \max(0, \nu - v(N \setminus \{i\}))$,
and to set the $2\sigma$ noise threshold.


**Algorithm**, for each prompt:
1. $n$ agent runs with the full set (we get answers $r_1, \dots, r_n$).
2. TF-IDF fitted on all $n$ answers at once, as in `analyze()`.
3. Cosine of $r_1$ with each of $r_2, \dots, r_n$: $r_1$ is the baseline, as in `analyze()`.

$\nu$ and $\sigma$ are the mean and standard deviation of these $n - 1$ numbers (over all prompts in the list). $\sigma$ is the spread of a single comparison, not the error of $\nu$: all comparisons are against the same $r_1$.

Here there are 3 prompts with 10 runs each on the full toolset. The first answer for each prompt is the baseline, the other nine are compared with it using the same metric as in `analyze()`.

In [3]:
NOISE_PROMPTS = [
    PROMPT,
    "What's the latest AI news and what is 17 plus 25?",
    "How is TSLA doing, and what's the weather in Tokyo?",
]
nf = measure_noise_floor(agent_shap, NOISE_PROMPTS, n_runs=10)
nf.to_json(OUT / "noise_floor.json")
print(nf)

noise floor  nu = 0.6653 ± 0.1079   (27 comparisons, 3 prompts, qwen2.5:3b)
bias       1 - nu = 0.3347
behavioural noise: 0.0% of runs with a different tool set


## Prompts where no tool is needed

In [4]:
NO_TOOL_PROMPTS = [
    "What is the capital of France?",
    "Who wrote Anna Karenina?",
    "Translate 'good morning' into Spanish.",
    "What does the acronym CPU stand for?",
    "Name three primary colors.",
]
coal, shap = [], []
for p in NO_TOOL_PROMPTS:
    res, sv = run(p)
    coal.append(res)
    shap.append(sv)

notool = pd.concat(coal, ignore_index=True)
notool.to_csv(OUT / "notool_coalitions.csv", index=False)
pd.DataFrame(shap).to_csv(OUT / "notool_shapley.csv", index=False)
print(pd.DataFrame(shap).set_index("prompt").round(4).to_string())
used = [c for c in notool.columns if c.startswith("used_")]
print("\ntool calls on these prompts, over all coalitions:")
print(notool.groupby("prompt", sort=False)[used].sum().to_string())


Analyzing 4 tools:
  - get_weather
  - get_stock_price
  - calculate
  - get_news

1. Getting baseline response (all tools)...



2. Testing tool combinations...



Testing tool combinations:   0%|          | 0/4 [00:00<?, ?it/s]


Testing tool combinations:  25%|██▌       | 1/4 [00:04<00:12,  4.19s/it]


Testing tool combinations:  50%|█████     | 2/4 [00:06<00:06,  3.35s/it]

...


Testing tool combinations:  50%|█████     | 2/4 [00:17<00:19,  9.69s/it]


Testing tool combinations:  75%|███████▌  | 3/4 [00:22<00:07,  7.42s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:28<00:00,  7.11s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:28<00:00,  7.17s/it]


3. Calculating similarities...

4. Computing Shapley values...

 Analysis complete!
                                        get_weather  get_stock_price  calculate  get_news
prompt                                                                                   
What is the capital of France?               0.0000           0.3381     0.3305    0.3314
Who wrote Anna Karenina?                     0.4989           0.0931     0.4080    0.0000
Translate 'good morning' into Spanish.       0.3993           0.2683     0.0000    0.3324
What does the acronym CPU stand for?         0.2034           0.4602     0.0000    0.3364
Name three primary colors.                   0.3012           0.4065     0.0000    0.2923

tool calls on these prompts, over all coalitions:
                                        used_get_news
prompt                                               
What is the capital of France?                    0.0
Who wrote Anna Karenina?                          0.0
Translate 'good mo

## Noise floor on each no-tool prompt

The similarity ceiling depends on the prompt. An answer with tool data is almost deterministic, while free text is not. So for the dummy check we always measure $\nu$ on the same prompts on which we compute the contributions.

In [5]:
import json
from dataclasses import asdict

nf_notool = {p: measure_noise_floor(agent_shap, [p], n_runs=10) for p in NO_TOOL_PROMPTS}
(OUT / "noise_floor_notool.json").write_text(
    json.dumps({p: asdict(nf) for p, nf in nf_notool.items()}, ensure_ascii=False, indent=2), encoding="utf-8")
for p, nf in nf_notool.items():
    print(f"{p:<40} nu = {nf.nu:.3f} ± {nf.sd:.3f}")

What is the capital of France?           nu = 0.316 ± 0.079
Who wrote Anna Karenina?                 nu = 0.318 ± 0.102
Translate 'good morning' into Spanish.   nu = 0.308 ± 0.104
What does the acronym CPU stand for?     nu = 0.424 ± 0.141
Name three primary colors.               nu = 0.274 ± 0.048
